# broadcast-initial-weights — faded example 1: Complete the parameter-sync broadcast loop

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-initial-weights`. The last cell reports your progress on the `Distributed: broadcast initial weights` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: broadcast initial weights` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-initial-weights`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-initial-weights"
DD_SUBTOPIC = "Distributed: broadcast initial weights"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To synchronize replicas at DDP startup you broadcast each `param.data` from rank 0. `dist.broadcast(tensor, src=0)` mutates the tensor storage in-place, so a simple loop over `model.parameters()` overwrites every divergent replica with rank 0's weights.

## Faded exercise 1

### Complete the broadcast loop

The `FakeDist` collective and the divergent models are already built. Complete `broadcast_params` so that, after the call, `rank1`'s weight matches `rank0`'s. You must iterate the model's parameters and broadcast each one from `src=0`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Linear(2, 2, bias=False)
    with t.no_grad():
        m.weight.fill_(float(rank + 1))
    return m

def broadcast_params(model, fake):
    fake.reset()
    for p in model.parameters():
        fake.broadcast(p.data, src=0)

rank0 = build_model(0)
fake = FakeDist([p.data.clone() for p in rank0.parameters()])
rank1 = build_model(1)
broadcast_params(rank1, fake)
print(rank1.weight.detach().flatten().tolist())


def _test():
    r0 = build_model(0)
    f = FakeDist([p.data.clone() for p in r0.parameters()])
    r1 = build_model(1)
    assert not t.equal(r1.weight.data, r0.weight.data), 'precondition: ranks should differ before broadcast'
    broadcast_params(r1, f)
    assert t.equal(r1.weight.data, r0.weight.data), 'rank1 weight must equal rank0 after broadcast'
    assert r1.weight.detach().flatten().tolist() == [1.0, 1.0, 1.0, 1.0]


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Linear(2, 2, bias=False)
    with t.no_grad():
        m.weight.fill_(float(rank + 1))
    return m

def broadcast_params(model, fake):
    fake.reset()
    for p in model.parameters():
        fake.broadcast(p.data, src=0)

rank0 = build_model(0)
fake = FakeDist([p.data.clone() for p in rank0.parameters()])
rank1 = build_model(1)
broadcast_params(rank1, fake)
print(rank1.weight.detach().flatten().tolist())
```
</details>